# Practical Lab 3 — Vanilla CNN and Fine-Tune VGG16 for Dogs vs Cats Classification

**Course:** CSCN8010 — Deep Learning  
**Author:** Muthu

---

## Introduction

This notebook demonstrates the core practice of Deep Learning Engineers: **take an existing model that does something similar to the task at hand, and fine-tune it for the specific problem.**

We tackle a binary image classification problem — distinguishing between photos of **dogs** and **cats** — using two approaches:

1. **Vanilla CNN** — A convolutional neural network trained from scratch with data augmentation and dropout regularization.
2. **Fine-Tuned VGG16** — A pre-trained VGG16 model (trained on ImageNet's 1.2 million images across 1,000 categories) fine-tuned for our specific binary classification task.

By comparing these two approaches, we demonstrate why transfer learning is the standard practice in modern deep learning — leveraging knowledge learned from large datasets to solve specific problems with limited data.

## 1. Imports and Setup

In [ ]:
import pathlib
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    ConfusionMatrixDisplay,
    auc,
)
from PIL import Image

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Obtain and Load the Data

The **Asirra Dogs vs Cats** dataset is a subset of the Microsoft Research Asirra dataset, originally containing 25,000 images. We use a curated subset of **5,000 images** organized into three splits:

| Split | Total Images | Cats | Dogs |
|-------|-------------|------|------|
| Training | 2,000 | 1,000 | 1,000 |
| Validation | 1,000 | 500 | 500 |
| Test | 2,000 | 1,000 | 1,000 |

The dataset is perfectly balanced — equal numbers of cats and dogs in each split — which means a random baseline would achieve 50% accuracy.

In [ ]:
# Define data paths
data_folder = pathlib.Path("./data/kaggle_dogs_vs_cats_small")

# Load datasets using Keras utility
# All images are resized to 180x180 pixels and batched in groups of 32
IMAGE_SIZE = (180, 180)
BATCH_SIZE = 32

train_dataset = keras.utils.image_dataset_from_directory(
    data_folder / "train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

validation_dataset = keras.utils.image_dataset_from_directory(
    data_folder / "validation",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

test_dataset = keras.utils.image_dataset_from_directory(
    data_folder / "test",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

# Display class names (alphabetical order: cat=0, dog=1)
class_names = train_dataset.class_names
print(f"Class names: {class_names}")
print(f"Class mapping: {class_names[0]}=0, {class_names[1]}=1")

In [ ]:
# Verify batch shapes
for images_batch, labels_batch in train_dataset.take(1):
    print(f"Images batch shape: {images_batch.shape}")  # (32, 180, 180, 3)
    print(f"Labels batch shape: {labels_batch.shape}")   # (32,)
    print(f"Pixel value range: [{images_batch.numpy().min()}, {images_batch.numpy().max()}]")
    print(f"Label values in batch: {np.unique(labels_batch.numpy())}")

## 3. Exploratory Data Analysis (EDA)

Before building any model, we need to understand what we're working with. EDA helps us identify potential issues — variable image sizes, unexpected color modes, class imbalances — that could affect model performance.

### 3a. Sample Images

In [ ]:
# Display a grid of 16 sample images (8 cats, 8 dogs) from the training set
plt.figure(figsize=(16, 8))

cat_count = 0
dog_count = 0
images_shown = []

for images_batch, labels_batch in train_dataset:
    for i in range(len(labels_batch)):
        label = labels_batch[i].numpy()
        if label == 0 and cat_count < 8:  # cat
            images_shown.append((images_batch[i], label))
            cat_count += 1
        elif label == 1 and dog_count < 8:  # dog
            images_shown.append((images_batch[i], label))
            dog_count += 1
        if cat_count >= 8 and dog_count >= 8:
            break
    if cat_count >= 8 and dog_count >= 8:
        break

for idx, (img, label) in enumerate(images_shown):
    plt.subplot(2, 8, idx + 1)
    plt.imshow(img.numpy().astype("uint8"))
    plt.title(class_names[label], fontsize=11)
    plt.axis("off")

plt.suptitle("Sample Images from Training Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3b. Class Distribution

We verify that the dataset is balanced across all splits — this is critical because an imbalanced dataset could lead a model to simply predict the majority class.

In [ ]:
# Count images per class in each split
splits = {"Train": data_folder / "train", "Validation": data_folder / "validation", "Test": data_folder / "test"}
counts = []

for split_name, split_path in splits.items():
    for class_name in ["cat", "dog"]:
        class_dir = split_path / class_name
        n_images = len(list(class_dir.glob("*.jpg")))
        counts.append({"Split": split_name, "Class": class_name.capitalize(), "Count": n_images})

df_counts = pd.DataFrame(counts)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_counts, x="Split", y="Count", hue="Class", palette=["steelblue", "coral"], ax=ax)
ax.set_title("Class Distribution Across Dataset Splits", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Images")

for container in ax.containers:
    ax.bar_label(container, fontsize=11)

plt.tight_layout()
plt.show()

# Print summary table
print(df_counts.pivot(index="Split", columns="Class", values="Count").to_string())

### 3c. Original Image Size Analysis

The raw images in the dataset come in many different sizes. Understanding the distribution of widths and heights helps us appreciate why resizing to a fixed dimension (180x180) is necessary.

In [ ]:
# Analyze original image sizes from raw files using PIL
widths, heights, aspect_ratios, color_modes = [], [], [], []

for split_path in [data_folder / "train", data_folder / "validation", data_folder / "test"]:
    for class_dir in sorted(split_path.iterdir()):
        for img_path in class_dir.glob("*.jpg"):
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
                    aspect_ratios.append(w / h)
                    color_modes.append(img.mode)
            except Exception:
                pass  # skip corrupted images

# Plot histograms of widths and heights
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(widths, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Image Widths", fontweight="bold")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Count")
axes[0].axvline(np.median(widths), color="red", linestyle="--", label=f"Median: {np.median(widths):.0f}")
axes[0].legend()

axes[1].hist(heights, bins=50, color="coral", edgecolor="black", alpha=0.7)
axes[1].set_title("Distribution of Image Heights", fontweight="bold")
axes[1].set_xlabel("Height (pixels)")
axes[1].set_ylabel("Count")
axes[1].axvline(np.median(heights), color="red", linestyle="--", label=f"Median: {np.median(heights):.0f}")
axes[1].legend()

axes[2].scatter(widths, heights, alpha=0.15, s=10, color="purple")
axes[2].set_title("Width vs Height (Aspect Ratio)", fontweight="bold")
axes[2].set_xlabel("Width (pixels)")
axes[2].set_ylabel("Height (pixels)")
axes[2].plot([0, max(widths)], [0, max(widths)], "r--", alpha=0.5, label="Square (1:1)")
axes[2].legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Width  — Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}, Median: {np.median(widths):.0f}")
print(f"Height — Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}, Median: {np.median(heights):.0f}")
print(f"Aspect Ratio — Mean: {np.mean(aspect_ratios):.2f}, Std: {np.std(aspect_ratios):.2f}")

### 3d. Pixel Intensity and Color Analysis

Understanding the pixel value distributions across RGB channels helps us verify that the images are standard color photos and identify any preprocessing needs.

In [ ]:
# Compute mean pixel values per channel across a sample of training images
channel_means_r, channel_means_g, channel_means_b = [], [], []

for images_batch, _ in train_dataset.take(10):  # sample 10 batches (~320 images)
    for img in images_batch:
        img_np = img.numpy()
        channel_means_r.append(img_np[:, :, 0].mean())
        channel_means_g.append(img_np[:, :, 1].mean())
        channel_means_b.append(img_np[:, :, 2].mean())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of mean pixel values per channel
axes[0].hist(channel_means_r, bins=40, alpha=0.6, color="red", label="Red", edgecolor="black")
axes[0].hist(channel_means_g, bins=40, alpha=0.6, color="green", label="Green", edgecolor="black")
axes[0].hist(channel_means_b, bins=40, alpha=0.6, color="blue", label="Blue", edgecolor="black")
axes[0].set_title("Mean Pixel Intensity Distribution per Channel", fontweight="bold")
axes[0].set_xlabel("Mean Pixel Value (0-255)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Show a sample image with its RGB channel decomposition
sample_img = None
for images_batch, labels_batch in train_dataset.take(1):
    sample_img = images_batch[0].numpy().astype("uint8")
    break

channel_names = ["Red Channel", "Green Channel", "Blue Channel"]
cmaps = ["Reds", "Greens", "Blues"]

# Show original
axes[1].imshow(sample_img)
axes[1].set_title("Sample Image (Original)", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Show channel decomposition separately
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(3):
    axes[i].imshow(sample_img[:, :, i], cmap=cmaps[i])
    axes[i].set_title(channel_names[i], fontweight="bold")
    axes[i].axis("off")
plt.suptitle("RGB Channel Decomposition", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3e. Dataset Summary Statistics

In [ ]:
# Summary statistics table
from collections import Counter
mode_counts = Counter(color_modes)

summary_data = {
    "Metric": [
        "Total Images",
        "Training Images",
        "Validation Images",
        "Test Images",
        "Classes",
        "Image Width (min / max / mean)",
        "Image Height (min / max / mean)",
        "Dominant Color Mode",
        "Non-RGB Images",
        "Mean Aspect Ratio",
    ],
    "Value": [
        len(widths),
        2000,
        1000,
        2000,
        "2 (cat, dog) — balanced",
        f"{min(widths)} / {max(widths)} / {np.mean(widths):.0f} px",
        f"{min(heights)} / {max(heights)} / {np.mean(heights):.0f} px",
        f"RGB ({mode_counts.get('RGB', 0)} images)",
        f"{len(widths) - mode_counts.get('RGB', 0)} images ({', '.join(k for k in mode_counts if k != 'RGB') or 'none'})",
        f"{np.mean(aspect_ratios):.2f} (std: {np.std(aspect_ratios):.2f})",
    ],
}
df_summary = pd.DataFrame(summary_data)
df_summary.style.hide(axis="index")

### EDA Insights

- **Balanced dataset**: Equal numbers of cats and dogs across all splits, so no class-weighting is needed and a random baseline would achieve 50% accuracy.
- **Variable image sizes**: Raw images range widely in resolution — the CNN requires fixed-size input (180x180), so all images are resized.
- **Aspect ratio distortion**: Resizing non-square images into a 180x180 square introduces some distortion, but CNNs are robust to mild geometric deformation.
- **Primarily RGB images**: The vast majority of images are standard 3-channel color photos. A small number may be grayscale, which Keras automatically converts to 3-channel during loading.
- **Real-world data challenges**: These are amateur pet photos with variable lighting, cluttered backgrounds, and different poses — representative of real-world computer vision problems.

---

## 4. Data Preprocessing

We define a data augmentation pipeline that creates random variations of training images. This effectively increases the training set size and helps the model generalize better by learning features that are invariant to horizontal flips, small rotations, and zoom levels.

In [ ]:
# Define the data augmentation pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),   # Flip left-right with 50% probability
    layers.RandomRotation(0.1),        # Rotate by up to ±10%
    layers.RandomZoom(0.2),            # Zoom in/out by up to 20%
])

# Visualize augmentation: show original image + 6 augmented versions
plt.figure(figsize=(16, 4))

# Get a sample image
for images_batch, labels_batch in train_dataset.take(1):
    sample_image = images_batch[0]
    sample_label = labels_batch[0].numpy()
    break

# Show original
plt.subplot(1, 7, 1)
plt.imshow(sample_image.numpy().astype("uint8"))
plt.title("Original", fontweight="bold")
plt.axis("off")

# Show 6 augmented versions
for i in range(6):
    augmented = data_augmentation(tf.expand_dims(sample_image, 0))
    plt.subplot(1, 7, i + 2)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.title(f"Augmented {i+1}")
    plt.axis("off")

plt.suptitle(f"Data Augmentation Examples ({class_names[sample_label]})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## 5. Model 1: Vanilla CNN (Trained from Scratch)

Our first model is a convolutional neural network built from scratch. The architecture follows a classic pattern of increasing filter depth through 5 convolutional blocks, with data augmentation and dropout for regularization.

**Architecture:**
- 5 convolutional layers with increasing filters (32 → 64 → 128 → 256 → 256)
- MaxPooling after each of the first 4 conv layers
- Data augmentation applied during training only
- Dropout (50%) before the output layer to reduce overfitting
- Binary sigmoid output for cat/dog classification

In [ ]:
# Build the Vanilla CNN model
inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 255)(x)

x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)

x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

vanilla_cnn = keras.Model(inputs=inputs, outputs=outputs)
vanilla_cnn.summary()

In [ ]:
# Compile and train the Vanilla CNN
vanilla_cnn.compile(
    loss="binary_crossentropy",
    optimizer="rmsprop",
    metrics=["accuracy"],
)

# Save the best model based on validation loss
vanilla_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="./models/vanilla_cnn_best.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]

vanilla_history = vanilla_cnn.fit(
    train_dataset,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=vanilla_callbacks,
)

In [ ]:
# Plot training curves for Vanilla CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(vanilla_history.history["accuracy"], label="Training Accuracy")
axes[0].plot(vanilla_history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Vanilla CNN — Accuracy", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(vanilla_history.history["loss"], label="Training Loss")
axes[1].plot(vanilla_history.history["val_loss"], label="Validation Loss")
axes[1].set_title("Vanilla CNN — Loss", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report best validation accuracy
best_val_epoch = np.argmin(vanilla_history.history["val_loss"])
print(f"Best validation loss at epoch {best_val_epoch + 1}")
print(f"  Val accuracy: {vanilla_history.history['val_accuracy'][best_val_epoch]:.4f}")
print(f"  Val loss: {vanilla_history.history['val_loss'][best_val_epoch]:.4f}")

---

## 6. Model 2: Fine-Tuned VGG16 (Transfer Learning)

Our second model leverages **transfer learning** — we take VGG16, a model pre-trained on ImageNet (1.2 million images, 1,000 categories), and fine-tune it for our specific binary classification task.

**Strategy:**
1. Load VGG16 with ImageNet weights, excluding the classification head
2. Freeze all layers except the last 4 (which learn task-specific features)
3. Add our own classification head (Dense + Dropout + Sigmoid)
4. Train with a very low learning rate (1e-5) to preserve pre-learned features

This approach works because the early layers of VGG16 already know how to detect edges, textures, and shapes — knowledge that transfers directly to distinguishing cats from dogs.

In [ ]:
# Load VGG16 pre-trained on ImageNet (without the top classification layers)
conv_base = keras.applications.vgg16.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(180, 180, 3),
)

# Freeze all layers, then unfreeze the last 4 for fine-tuning
conv_base.trainable = True
for layer in conv_base.layers[:-4]:
    layer.trainable = False

# Show which layers are trainable
print("VGG16 Layer Trainability:")
print("-" * 50)
for layer in conv_base.layers:
    print(f"  {layer.name:25s} trainable={layer.trainable}")

In [ ]:
# Build the fine-tuned VGG16 model
inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = keras.applications.vgg16.preprocess_input(x)  # VGG16-specific preprocessing
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

vgg16_model = keras.Model(inputs=inputs, outputs=outputs)
vgg16_model.summary()

In [ ]:
# Compile with a very low learning rate to preserve pre-trained features
vgg16_model.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
    metrics=["accuracy"],
)

# Save the best model based on validation loss
vgg16_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="./models/vgg16_finetuned_best.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]

vgg16_history = vgg16_model.fit(
    train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=vgg16_callbacks,
)

In [ ]:
# Plot training curves for VGG16
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(vgg16_history.history["accuracy"], label="Training Accuracy")
axes[0].plot(vgg16_history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Fine-Tuned VGG16 — Accuracy", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(vgg16_history.history["loss"], label="Training Loss")
axes[1].plot(vgg16_history.history["val_loss"], label="Validation Loss")
axes[1].set_title("Fine-Tuned VGG16 — Loss", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report best validation accuracy
best_val_epoch = np.argmin(vgg16_history.history["val_loss"])
print(f"Best validation loss at epoch {best_val_epoch + 1}")
print(f"  Val accuracy: {vgg16_history.history['val_accuracy'][best_val_epoch]:.4f}")
print(f"  Val loss: {vgg16_history.history['val_loss'][best_val_epoch]:.4f}")

---

## 7. Model Comparison and Evaluation

We now load the **best version** of each model (saved by ModelCheckpoint) and evaluate them on the held-out test set. We compare:
- **Accuracy** — Overall correctness
- **Confusion Matrix** — True/false positives and negatives
- **Precision, Recall, F1-Score** — Per-class performance
- **Precision-Recall Curve** — Trade-off visualization
- **Misclassified Examples** — Understanding failure modes

### 7.0 Load Best Models and Generate Predictions

In [ ]:
# Load the best saved models
best_vanilla = keras.models.load_model("./models/vanilla_cnn_best.keras")
best_vgg16 = keras.models.load_model("./models/vgg16_finetuned_best.keras")

# Collect all test labels and images
test_images_list = []
test_labels_list = []
for images_batch, labels_batch in test_dataset:
    test_images_list.append(images_batch.numpy())
    test_labels_list.append(labels_batch.numpy())

test_images_all = np.concatenate(test_images_list)
test_labels_all = np.concatenate(test_labels_list)

# Generate predictions (raw probabilities)
vanilla_probs = best_vanilla.predict(test_images_all).flatten()
vgg16_probs = best_vgg16.predict(test_images_all).flatten()

# Convert to class predictions (threshold = 0.5)
vanilla_preds = (vanilla_probs >= 0.5).astype(int)
vgg16_preds = (vgg16_probs >= 0.5).astype(int)

print(f"Test set size: {len(test_labels_all)} images")
print(f"Vanilla CNN predictions generated: {len(vanilla_preds)}")
print(f"VGG16 predictions generated: {len(vgg16_preds)}")

### 7a. Accuracy Comparison

In [ ]:
# Calculate test accuracy for both models
vanilla_acc = np.mean(vanilla_preds == test_labels_all)
vgg16_acc = np.mean(vgg16_preds == test_labels_all)

# Side-by-side bar chart
fig, ax = plt.subplots(figsize=(8, 5))
models = ["Vanilla CNN", "Fine-Tuned VGG16"]
accuracies = [vanilla_acc, vgg16_acc]
colors = ["steelblue", "coral"]

bars = ax.bar(models, accuracies, color=colors, width=0.5, edgecolor="black")
ax.set_ylabel("Test Accuracy")
ax.set_title("Model Accuracy Comparison on Test Set", fontweight="bold")
ax.set_ylim(0, 1.0)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Random Baseline (50%)")
ax.legend()

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.2%}", ha="center", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Vanilla CNN Test Accuracy:      {vanilla_acc:.4f} ({vanilla_acc:.2%})")
print(f"Fine-Tuned VGG16 Test Accuracy: {vgg16_acc:.4f} ({vgg16_acc:.2%})")
print(f"Improvement: +{(vgg16_acc - vanilla_acc):.2%}")

### 7b. Confusion Matrix

In [ ]:
# Confusion matrices for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in [
    (axes[0], vanilla_preds, "Vanilla CNN"),
    (axes[1], vgg16_preds, "Fine-Tuned VGG16"),
]:
    cm = confusion_matrix(test_labels_all, preds)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix — {title}", fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

plt.tight_layout()
plt.show()

### 7c. Precision, Recall, and F1-Score

In [ ]:
# Classification reports for both models
print("=" * 60)
print("VANILLA CNN — Classification Report")
print("=" * 60)
print(classification_report(test_labels_all, vanilla_preds, target_names=class_names))

print("\n")
print("=" * 60)
print("FINE-TUNED VGG16 — Classification Report")
print("=" * 60)
print(classification_report(test_labels_all, vgg16_preds, target_names=class_names))

In [ ]:
# Side-by-side comparison as a DataFrame
from sklearn.metrics import precision_score, recall_score, f1_score

metrics_data = {
    "Metric": ["Accuracy", "Precision (Cat)", "Precision (Dog)",
                "Recall (Cat)", "Recall (Dog)", "F1-Score (Cat)", "F1-Score (Dog)"],
    "Vanilla CNN": [
        f"{vanilla_acc:.4f}",
        f"{precision_score(test_labels_all, vanilla_preds, pos_label=0):.4f}",
        f"{precision_score(test_labels_all, vanilla_preds, pos_label=1):.4f}",
        f"{recall_score(test_labels_all, vanilla_preds, pos_label=0):.4f}",
        f"{recall_score(test_labels_all, vanilla_preds, pos_label=1):.4f}",
        f"{f1_score(test_labels_all, vanilla_preds, pos_label=0):.4f}",
        f"{f1_score(test_labels_all, vanilla_preds, pos_label=1):.4f}",
    ],
    "Fine-Tuned VGG16": [
        f"{vgg16_acc:.4f}",
        f"{precision_score(test_labels_all, vgg16_preds, pos_label=0):.4f}",
        f"{precision_score(test_labels_all, vgg16_preds, pos_label=1):.4f}",
        f"{recall_score(test_labels_all, vgg16_preds, pos_label=0):.4f}",
        f"{recall_score(test_labels_all, vgg16_preds, pos_label=1):.4f}",
        f"{f1_score(test_labels_all, vgg16_preds, pos_label=0):.4f}",
        f"{f1_score(test_labels_all, vgg16_preds, pos_label=1):.4f}",
    ],
}

df_metrics = pd.DataFrame(metrics_data)
df_metrics.style.hide(axis="index")

### 7d. Precision-Recall Curve

In [ ]:
# Precision-Recall curves for both models
fig, ax = plt.subplots(figsize=(10, 7))

# Vanilla CNN PR curve
vanilla_precision, vanilla_recall, _ = precision_recall_curve(test_labels_all, vanilla_probs)
vanilla_auc_pr = auc(vanilla_recall, vanilla_precision)
ax.plot(vanilla_recall, vanilla_precision, color="steelblue", linewidth=2,
        label=f"Vanilla CNN (AUC-PR = {vanilla_auc_pr:.3f})")

# VGG16 PR curve
vgg16_precision, vgg16_recall, _ = precision_recall_curve(test_labels_all, vgg16_probs)
vgg16_auc_pr = auc(vgg16_recall, vgg16_precision)
ax.plot(vgg16_recall, vgg16_precision, color="coral", linewidth=2,
        label=f"Fine-Tuned VGG16 (AUC-PR = {vgg16_auc_pr:.3f})")

# Mark the threshold=0.5 operating points
vanilla_pred_at_05 = (vanilla_probs >= 0.5).astype(int)
vgg16_pred_at_05 = (vgg16_probs >= 0.5).astype(int)

v_prec = precision_score(test_labels_all, vanilla_pred_at_05)
v_rec = recall_score(test_labels_all, vanilla_pred_at_05)
ax.scatter([v_rec], [v_prec], color="steelblue", s=100, zorder=5,
           edgecolors="black", marker="o", label=f"Vanilla CNN @ t=0.5")

vg_prec = precision_score(test_labels_all, vgg16_pred_at_05)
vg_rec = recall_score(test_labels_all, vgg16_pred_at_05)
ax.scatter([vg_rec], [vg_prec], color="coral", s=100, zorder=5,
           edgecolors="black", marker="o", label=f"VGG16 @ t=0.5")

# Random baseline
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Random Baseline")

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve — Model Comparison", fontsize=14, fontweight="bold")
ax.legend(loc="lower left", fontsize=10)
ax.set_xlim([0, 1.02])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7e. Misclassified Examples — Error Analysis

Understanding *where* a model fails is often more valuable than knowing its overall accuracy. We display images that each model misclassified, along with the true label, predicted label, and the model's confidence.

In [ ]:
def show_misclassified(images, true_labels, pred_labels, probs, model_name, n=10):
    """Display misclassified images with true/predicted labels and confidence."""
    misclassified_idx = np.where(pred_labels != true_labels)[0]
    n_show = min(n, len(misclassified_idx))

    if n_show == 0:
        print(f"{model_name}: No misclassified images!")
        return

    # Pick a random sample of misclassified images
    rng = np.random.default_rng(42)
    selected = rng.choice(misclassified_idx, size=n_show, replace=False)

    fig, axes = plt.subplots(2, 5, figsize=(18, 8))
    fig.suptitle(f"{model_name} — Misclassified Examples ({len(misclassified_idx)} total errors)",
                 fontsize=14, fontweight="bold")

    for i, idx in enumerate(selected):
        row, col = i // 5, i % 5
        ax = axes[row, col]
        ax.imshow(images[idx].astype("uint8"))

        true_label = class_names[int(true_labels[idx])]
        pred_label = class_names[int(pred_labels[idx])]
        confidence = probs[idx] if pred_labels[idx] == 1 else 1 - probs[idx]

        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                     fontsize=10, color="red", fontweight="bold")
        ax.axis("off")

    # Hide empty subplots
    for i in range(n_show, 10):
        row, col = i // 5, i % 5
        axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

# Show misclassified examples for both models
show_misclassified(test_images_all, test_labels_all, vanilla_preds, vanilla_probs, "Vanilla CNN")
show_misclassified(test_images_all, test_labels_all, vgg16_preds, vgg16_probs, "Fine-Tuned VGG16")

In [ ]:
# Error analysis summary
vanilla_errors = np.sum(vanilla_preds != test_labels_all)
vgg16_errors = np.sum(vgg16_preds != test_labels_all)

# Breakdown: which class is harder for each model?
vanilla_cat_errors = np.sum((vanilla_preds != test_labels_all) & (test_labels_all == 0))
vanilla_dog_errors = np.sum((vanilla_preds != test_labels_all) & (test_labels_all == 1))
vgg16_cat_errors = np.sum((vgg16_preds != test_labels_all) & (test_labels_all == 0))
vgg16_dog_errors = np.sum((vgg16_preds != test_labels_all) & (test_labels_all == 1))

error_df = pd.DataFrame({
    "Model": ["Vanilla CNN", "Fine-Tuned VGG16"],
    "Total Errors": [vanilla_errors, vgg16_errors],
    "Cat Misclassified as Dog": [vanilla_cat_errors, vgg16_cat_errors],
    "Dog Misclassified as Cat": [vanilla_dog_errors, vgg16_dog_errors],
    "Error Rate": [f"{vanilla_errors/len(test_labels_all):.2%}", f"{vgg16_errors/len(test_labels_all):.2%}"],
})
error_df.style.hide(axis="index")

---

## 8. Conclusions

### Key Findings

1. **Transfer learning significantly outperforms training from scratch.** The fine-tuned VGG16 model achieves substantially higher accuracy than the vanilla CNN trained from scratch, despite using the same training data (2,000 images). This demonstrates the power of leveraging knowledge from large-scale pre-training (ImageNet: 1.2M images).

2. **The vanilla CNN, while limited, still learns meaningful patterns.** With data augmentation and dropout regularization, it surpasses the 50% random baseline by a significant margin, proving that even a simple architecture can learn to distinguish visual features with limited data.

3. **Regularization is essential for small datasets.** Both data augmentation (random flips, rotations, zoom) and dropout (50%) were critical for preventing overfitting. Without these techniques, the vanilla CNN would memorize training data and fail to generalize.

4. **Fine-tuning strategy matters.** By freezing the early VGG16 layers (which capture universal features like edges and textures) and only fine-tuning the last 4 layers (which capture task-specific features), we achieve optimal performance with a very low learning rate (1e-5) to avoid destroying pre-learned representations.

5. **Error analysis reveals model limitations.** Both models struggle with images that have cluttered backgrounds, unusual poses, partial views, or multiple subjects. The VGG16 model makes fewer errors overall and tends to be more confident in its correct predictions.

### Practical Takeaway

> **The core practice of Deep Learning Engineers**: Don't train from scratch. Find an existing model that does something similar — in this case, VGG16 trained on ImageNet includes cats and dogs among its 1,000 categories — and fine-tune it for your specific task. This approach delivers better results with less data, less compute, and less time.

### Potential Improvements

- **More training data**: The full 25,000-image dataset would likely improve both models.
- **Learning rate scheduling**: Reducing the learning rate over time could help both models converge better.
- **Modern architectures**: ResNet, EfficientNet, or Vision Transformers could outperform VGG16.
- **Ensemble methods**: Combining predictions from both models could reduce overall errors.
- **Test-time augmentation**: Averaging predictions over multiple augmented versions of each test image.